<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v4/vision/roi-align/roi-align.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("roi-align", ...)`


# RoIAlign (and RoIPool) from Scratch

**Difficulty**: 🔴 Hard

### Problem Statement

A two-stage detector (Faster R-CNN, Mask R-CNN, Cascade R-CNN) first proposes boxes and then runs a small head on a **fixed-size feature crop for every box**. torchvision's Faster R-CNN pools a 7×7 crop per box for the box head, and its Mask R-CNN and Keypoint R-CNN pool 14×14 crops for the mask and keypoint heads (`MultiScaleRoIAlign(..., sampling_ratio=2)`). Boxes are given in image pixels, but the features come from a map with stride 4–32, so the crop has to be read at fractional positions of a coarse grid.

**RoIPool** (Fast R-CNN, Girshick 2015) does this with two roundings: it snaps the box to the feature grid, splits it into integer-edged bins and max-pools each bin. At stride 16 the first rounding alone can move the crop by up to 8 image pixels. That hardly matters for classification, but it does matter for a 28×28 mask or a keypoint heatmap predicted in the crop's coordinates. **RoIAlign** (Mask R-CNN, He et al. 2017) removes every rounding: it takes bilinear samples at regular points inside each bin and averages them. In Mask R-CNN's ablation this change alone raised mask AP from 26.9 to 30.3 with stride-16 features (Table 2c), raised it by 7.3 AP with stride-32 features (Table 2d), and raised keypoint AP by 4.4 (Table 6).

Implement both, matching `torchvision.ops.roi_align` and `torchvision.ops.roi_pool` **exactly**, including their border and rounding conventions. You already wrote the clamped two-tap bilinear gather in **bilinear-interpolation**. This problem is about the geometry around that gather: where the samples go, which ones count, and how they are averaged.

Coordinates: `x` is the width (column) axis and `y` the height (row) axis. A box row is `[batch_index, x1, y1, x2, y2]` in **image** pixels, and `spatial_scale` $s$ maps it to the feature map (e.g. $s = 1/16$ for a stride-16 map).

**RoIAlign for one box.** The y axis is shown below; the x axis is identical with $W$ and $\text{out}_w$.

1. **Scale and shift, with no rounding:** $y_1' = y_1 s - o$ and $y_2' = y_2 s - o$, where $o = 0.5$ if `aligned` else $0$.
2. **Size:** $h = y_2' - y_1'$. Only when `aligned=False`, $h \leftarrow \max(h, 1)$ (torchvision forces malformed RoIs to 1×1).
3. **Bins and grid:** the bin size is $b = h / \text{out}_h$. The number of samples per bin along this axis is $g = $ `sampling_ratio` if it is $> 0$, else the **adaptive** $g = \lceil h / \text{out}_h \rceil$. It is computed per box and per axis, and it can be $0$ (a zero-height box with `aligned=True`).
4. **Sample positions:** sample $i$ of bin $p$ is at the centre of the $i$-th of $g$ equal sub-cells,
$$y_{p,i} = y_1' + p\,b + \left(i + \tfrac12\right)\frac{b}{g}, \qquad i = 0, \dots, g - 1.$$
5. **Reading a sample** at $(y, x)$ from $F = $ `input[batch_index]`. If $y < -1$, $y > H$, $x < -1$ or $x > W$, the sample is $0$. Otherwise clamp $y$ to $[0, H-1]$ and $x$ to $[0, W-1]$, then interpolate bilinearly between rows $\lfloor y \rfloor$ and $\min(\lfloor y \rfloor + 1, H - 1)$, and likewise for columns. So a sample less than one pixel outside the map reads the border pixel at **full** weight, and a sample further out is dropped but **still counts** in the denominator.
6. **Average:**
$$\text{out}[k, :, p, q] = \frac{1}{\max(g_h\, g_w,\ 1)} \sum_{i=0}^{g_h - 1} \sum_{j=0}^{g_w - 1} \text{sample}(y_{p,i},\ x_{q,j}).$$

`aligned=True` (Detectron2's "ROIAlignV2") uses the half-pixel pixel model from `align_corners=False`. Pixel $i$ covers $[i, i+1)$ and its centre is at $i + \tfrac12$, so a continuous box coordinate $c$ corresponds to index $c - \tfrac12$. With `aligned=False` (the legacy default, which torchvision's detection models still use) every sample sits half a pixel off.

**RoIPool for one box.**

1. Snap the box to the grid: $x_1^r = \operatorname{round}(x_1 s)$, and likewise for $y_1, x_2, y_2$. This is C's `round`, which sends halves **away from zero** ($2.5 \to 3$, $-0.5 \to -1$). `torch.round` and Python's `round` send halves to the nearest even integer instead.
2. Size: $w = \max(x_2^r - x_1^r + 1,\ 1)$, and the same for $h$. The `+1` is the legacy inclusive-end convention.
3. Bins: $b_h = h / \text{out}_h$ is a float. Bin $p$ covers rows $[\lfloor p\, b_h \rfloor,\ \lceil (p+1)\, b_h \rceil) + y_1^r$, and both ends are then clipped to $[0, H]$. Columns work the same way.
4. The output is the **max** over the bin's pixels, or $0$ if the clipped bin is empty.

---

### Requirements

1. **`roi_align(input, boxes, output_size, spatial_scale=1.0, sampling_ratio=-1, aligned=False) -> Tensor`**
   - `input`: `(N, C, H, W)`, float32 or float64. `boxes`: `(K, 5)` with the same dtype, rows `[batch_index, x1, y1, x2, y2]` in image coordinates with $x_1 \le x_2$, $y_1 \le y_2$. Boxes may stick out of the image or lie completely outside it. `output_size = (out_h, out_w)`.
   - Return `(K, C, out_h, out_w)` with `input`'s dtype. With `K = 0`, return an empty `(0, C, out_h, out_w)` tensor.
   - Match `torchvision.ops.roi_align` to `1e-8` in float64 (`2e-4` in float32) for both values of `aligned`, fixed and adaptive `sampling_ratio`, any `spatial_scale`, several images per batch, and tiny, zero-size, partly-outside and fully-outside boxes.
   - Never return NaN. An empty sampling grid gives $0$.
   - Differentiable with respect to `input`, with the same gradient as torchvision: each input pixel collects `weight × upstream / count` from every sample of every box that reads it. No gradient with respect to `boxes` is required.
   - Vectorized: a Python loop over the K boxes is fine, because the adaptive grid size differs per box. Do **not** loop over channels, bins, sampling points or pixels.
2. **`roi_pool(input, boxes, output_size, spatial_scale=1.0) -> Tensor`** has the same arguments and shapes and must match `torchvision.ops.roi_pool` exactly. Loops over boxes and bins are fine; vectorize over channels.
   - Floating-point subtlety: torchvision computes $\lfloor p \cdot (h/\text{out}) \rfloor$ and $\lceil (p+1)(h/\text{out}) \rceil$ in floating point. For a few (size, out) pairs this differs from exact integer arithmetic, e.g. $\lceil 7 \cdot (29/7) \rceil = 30$ in float64. The grader avoids those pairs, so either style passes.

---

### Constraints

- ✅ Pure PyTorch: `torch.arange`, `floor`, `clamp`, advanced indexing, `gather`, `index_select`, `einsum`, `scatter_add_`.
- ✅ `F.grid_sample` is allowed for the bilinear step, but only if you reproduce the clamp-then-cut-off border rule and convert pixel coordinates to its normalised coordinates with the formula that matches its `align_corners` flag.
- ✅ Converting the **box** coordinates to Python floats (`boxes.tolist()`) is fine. The gradient only has to flow to `input`.
- ❌ Do not call `torchvision.ops` (only the optional cross-check cell imports it).
- ❌ No Python loop over channels, bins or sampling points in `roi_align`. The grader counts torch calls for the same 3 boxes at C=1 / 2×2 output / 1 sample per bin and at C=6 / 7×7 / 3×3 samples, and the two counts must be about the same.
- ❌ No `.item()`, `.tolist()` or numpy on the **feature values** in `roi_align` (the gradient has to flow).

---

<details>
  <summary>💡 Hint</summary>

  - Draw one axis: a box from $y_1'$ to $y_2'$ split into `out_h` bins, each with `g` samples at the centres of `g` equal sub-cells. Where is sample 0 of bin 0? Where is the last sample of the last bin?
  - The samples of one box form a tensor product: every y position is paired with every x position. Build the y positions as one vector of length `out_h * g_h` and the x positions as one vector of length `out_w * g_w`, then reuse your separable two-tap gather from bilinear-interpolation. What shape do you get before averaging, and which axes do you sum?
  - How can an out-of-range sample contribute 0 without breaking the gather? Think about its weights, not its indices.
  - What does `sampling_ratio=-1` give for a zero-width box with `aligned=True`, and what do you divide by then?
  - For `roi_pool`, try `torch.round(torch.tensor([0.5, 1.5, 2.5]))` and compare it with the rule above.

</details>

In [ ]:
import math

import torch
import torch.nn.functional as F

In [ ]:
torch.manual_seed(0)
torch.set_printoptions(precision=3, linewidth=120)

# A linear "ramp" feature map: value = 10 * y + x at pixel (y, x). Bilinear interpolation of a
# linear function is exact, and the samples of a bin are symmetric about the bin centre, so for a
# box whose samples all land inside the map, RoIAlign returns the ramp at each bin centre.
H, W = 8, 10
ramp = (10 * torch.arange(H, dtype=torch.float64).view(H, 1)
        + torch.arange(W, dtype=torch.float64).view(1, W)).view(1, 1, H, W)
print(ramp[0, 0])

# A batch of 2 stride-4 feature maps for a 48 x 64 image, and some boxes in image pixels.
feats = torch.randn(2, 4, 12, 16, dtype=torch.float64)
boxes = torch.tensor([[0, 4.0, 6.0, 30.0, 40.0],     # [batch_index, x1, y1, x2, y2]
                      [1, 10.0, 2.0, 60.0, 20.0],
                      [0, -8.0, 30.0, 20.0, 56.0]],  # sticks out of the left and bottom edges
                     dtype=torch.float64)

In [ ]:
def roi_align(input: torch.Tensor, boxes: torch.Tensor, output_size: tuple,
              spatial_scale: float = 1.0, sampling_ratio: int = -1,
              aligned: bool = False) -> torch.Tensor:
    """
    RoIAlign with torchvision.ops.roi_align semantics.

    Args:
        input: (N, C, H, W) feature map.
        boxes: (K, 5) rows [batch_index, x1, y1, x2, y2] in input-image coordinates.
        output_size: (out_h, out_w).
        spatial_scale: feature-map size / image size (e.g. 1/16 for a stride-16 map).
        sampling_ratio: samples per bin along each axis; <= 0 means adaptive ceil(roi / out).
        aligned: subtract 0.5 after scaling (half-pixel pixel model).

    Returns:
        (K, C, out_h, out_w) tensor with input's dtype, differentiable w.r.t. input.
    """
    # TODO: loop over boxes; map the box to feature coordinates (scale, then shift if aligned)
    # TODO: roi size (min 1 if not aligned), bin size, grid size per axis (fixed or adaptive)
    # TODO: sample positions per axis -> low/high tap indices and weights (0 beyond the cut-off)
    # TODO: separable gather (rows, then columns), sum each bin's samples, divide by max(count, 1)
    # TODO: handle K == 0
    raise NotImplementedError

In [ ]:
# Test 1: shapes, dtype, no boxes
out = roi_align(feats, boxes, (7, 7), spatial_scale=0.25)
assert out.shape == (3, 4, 7, 7), f"expected (3, 4, 7, 7), got {tuple(out.shape)}"
assert out.dtype == torch.float64
assert roi_align(feats.float(), boxes.float(), (2, 3), spatial_scale=0.25).dtype == torch.float32
empty = roi_align(feats, torch.zeros(0, 5, dtype=torch.float64), (7, 7))
assert empty.shape == (0, 4, 7, 7), f"no boxes -> (0, C, 7, 7), got {tuple(empty.shape)}"
print("Test 1 passed: shapes and dtypes")

In [ ]:
# Test 2: on the linear ramp, each bin equals the ramp at its centre
# Box x in [2, 6], y in [1, 5], 2x2 bins of size 2: bin centres y in {2, 4}, x in {3, 5}.
box = torch.tensor([[0, 2.0, 1.0, 6.0, 5.0]], dtype=torch.float64)
for sr in (1, 2, 3, -1):
    got = roi_align(ramp, box, (2, 2), sampling_ratio=sr, aligned=False)[0, 0]
    assert torch.allclose(got, torch.tensor([[23.0, 25.0], [43.0, 45.0]], dtype=torch.float64)), \
        f"aligned=False, sampling_ratio={sr}: got\n{got}"
    # aligned=True shifts every sample by -0.5 pixel in y and x: 10 * (-0.5) + (-0.5) = -5.5
    got = roi_align(ramp, box, (2, 2), sampling_ratio=sr, aligned=True)[0, 0]
    assert torch.allclose(got, torch.tensor([[17.5, 19.5], [37.5, 39.5]], dtype=torch.float64)), \
        f"aligned=True, sampling_ratio={sr}: got\n{got}"
# spatial_scale: the same box in a 2x larger image
got = roi_align(ramp, box * torch.tensor([1, 2, 2, 2, 2]), (2, 2), spatial_scale=0.5)[0, 0]
assert torch.allclose(got, torch.tensor([[23.0, 25.0], [43.0, 45.0]], dtype=torch.float64))
print("Test 2 passed: sample positions, aligned shift, spatial_scale")

In [ ]:
# Test 3: the border rule. A 1x1-pixel box with sampling_ratio=1 samples its centre.
def centre(yc, xc, img=ramp):
    b = torch.tensor([[0, xc - 0.5, yc - 0.5, xc + 0.5, yc + 0.5]], dtype=torch.float64)
    return roi_align(img, b, (1, 1), sampling_ratio=1).item()

assert centre(-0.5, 3.0) == 3.0, "-1 <= y < 0 reads row 0 at full weight (clamp, no zero padding)"
assert centre(-1.0, 3.0) == 3.0, "y = -1 exactly is still clamped to row 0"
assert centre(-1.5, 3.0) == 0.0, "y < -1 contributes 0"
assert centre(8.0, 3.0) == 73.0, "y = H exactly reads row H - 1"
assert centre(8.5, 3.0) == 0.0, "y > H contributes 0"
assert centre(2.0, -0.75) == 20.0 and centre(2.0, 10.25) == 0.0, "same rule along x"
# A box entirely outside the map gives zeros (not NaN).
far = torch.tensor([[0, 30.0, 30.0, 40.0, 40.0]], dtype=torch.float64)
assert roi_align(ramp, far, (2, 2)).abs().max().item() == 0.0
# aligned=True with a zero-width box and an adaptive grid: ceil(0 / out) = 0 samples -> 0
flat = torch.tensor([[0, 3.0, 1.0, 3.0, 6.0]], dtype=torch.float64)
z = roi_align(ramp, flat, (2, 2), sampling_ratio=-1, aligned=True)
assert torch.isfinite(z).all() and z.abs().max().item() == 0.0, f"expected zeros, got {z}"
print("Test 3 passed: border rule and empty grids")

In [ ]:
# Test 4: adaptive grid, batch index
# This box is 5 x 5.5 feature pixels -> with a 2x2 output the adaptive grid is ceil(2.5) x ceil(2.75) = 3 x 3.
b = torch.tensor([[1, 1.0, 2.0, 6.5, 7.0]], dtype=torch.float64)
adaptive = roi_align(feats, b, (2, 2), sampling_ratio=-1)
assert torch.allclose(adaptive, roi_align(feats, b, (2, 2), sampling_ratio=3)), \
    "sampling_ratio=-1 should use ceil(roi / out) = 3 samples per axis here"
assert not torch.allclose(adaptive, roi_align(feats, b, (2, 2), sampling_ratio=2)), \
    "adaptive must not be a fixed 2x2 grid"
assert torch.allclose(roi_align(feats, b, (2, 2), sampling_ratio=0), adaptive), "0 also means adaptive"
# batch index 1 reads feats[1]
b0 = b.clone(); b0[0, 0] = 0
assert torch.allclose(adaptive, roi_align(feats[1:], b0, (2, 2))), "boxes[:, 0] selects the image"
print("Test 4 passed: adaptive sampling grid and batch index")

In [ ]:
# Test 5: gradients. For boxes whose samples all land inside the map, every output bin's
# weights sum to 1, so with an upstream gradient of ones the input gradient sums to K*out_h*out_w
# in every channel.
x = feats.clone().requires_grad_(True)
inside = torch.tensor([[0, 4.0, 4.0, 40.0, 30.0], [1, 8.0, 10.0, 50.0, 40.0], [1, 8.0, 10.0, 50.0, 40.0]],
                      dtype=torch.float64)
roi_align(x, inside, (3, 3), spatial_scale=0.25, sampling_ratio=2, aligned=True).sum().backward()
assert x.grad is not None and torch.isfinite(x.grad).all(), "no gradient reached the input"
assert abs(x.grad.sum().item() - 3 * 3 * 3 * 4) < 1e-9, x.grad.sum().item()  # K * out_h * out_w * C
# adding a box that lies entirely outside the map does not change the gradient
far_box = torch.tensor([[0, 200.0, 200.0, 220.0, 220.0]], dtype=torch.float64)
x.grad = None
roi_align(x, torch.cat([inside[:1], far_box]), (2, 2), 0.25).sum().backward()
g_both = x.grad.clone()
x.grad = None
roi_align(x, inside[:1], (2, 2), 0.25).sum().backward()
assert torch.allclose(g_both, x.grad), "samples beyond the cut-off must not receive gradient"
print("Test 5 passed: gradients")

In [ ]:
def roi_pool(input: torch.Tensor, boxes: torch.Tensor, output_size: tuple,
             spatial_scale: float = 1.0) -> torch.Tensor:
    """
    RoIPool (Fast R-CNN) with torchvision.ops.roi_pool semantics.

    Args:
        input: (N, C, H, W) feature map.
        boxes: (K, 5) rows [batch_index, x1, y1, x2, y2] in input-image coordinates.
        output_size: (out_h, out_w).
        spatial_scale: feature-map size / image size.

    Returns:
        (K, C, out_h, out_w): max over each quantized bin, 0 for an empty bin.
    """
    # TODO: round each scaled coordinate half AWAY from zero
    # TODO: roi size = end - start + 1 (at least 1); float bin size
    # TODO: bin p: [floor(p * bin), ceil((p + 1) * bin)) + start, clipped to [0, H] / [0, W]
    # TODO: max over the bin (vectorized over channels); empty bin -> 0
    raise NotImplementedError

In [ ]:
# Test 6: RoIPool by hand on a 6x7 map whose value is 7 * y + x
img = torch.arange(42, dtype=torch.float64).view(1, 1, 6, 7)
# Box (x 1..5, y 1..4): roi_w = 5 - 1 + 1 = 5, roi_h = 4 - 1 + 1 = 4; 2x2 bins of 2 x 2.5 pixels:
# rows [1, 3) and [3, 5); columns [1, 4) and [3, 6) (they overlap). Max = bottom-right of each.
got = roi_pool(img, torch.tensor([[0, 1.0, 1.0, 5.0, 4.0]], dtype=torch.float64), (2, 2))[0, 0]
assert torch.equal(got, torch.tensor([[17.0, 19.0], [31.0, 33.0]], dtype=torch.float64)), got
# Rounding: x1 = y1 = 1 and x2 = y2 = 5 at scale 0.5 are 0.5 and 2.5 -> C round gives 1 and 3.
got = roi_pool(img, torch.tensor([[0, 1.0, 1.0, 5.0, 5.0]], dtype=torch.float64), (1, 1), 0.5)
assert got.item() == 24.0, f"expected max over rows/cols 1..3 = 24 (round half away from zero), got {got.item()}"
# Empty bins (box off the map) are 0
off = roi_pool(img, torch.tensor([[0, 10.0, 10.0, 14.0, 12.0]], dtype=torch.float64), (2, 2))
assert off.abs().max().item() == 0.0, off
print("Test 6 passed: roi_pool")

In [ ]:
# Optional: cross-check against torchvision when it is installed (it is on Colab).
try:
    from torchvision.ops import roi_align as tv_roi_align, roi_pool as tv_roi_pool
except ImportError:
    print("torchvision not installed -- skipping the cross-check")
else:
    g = torch.Generator().manual_seed(1)
    fm = torch.randn(2, 3, 13, 17, generator=g, dtype=torch.float64)
    xy = torch.rand(40, 2, generator=g, dtype=torch.float64) * torch.tensor([90.0, 70.0]) - 10
    wh = torch.rand(40, 2, generator=g, dtype=torch.float64) * 40
    rois = torch.cat([torch.randint(0, 2, (40, 1), generator=g).double(), xy, xy + wh], 1)
    for sr in (-1, 2):
        for al in (False, True):
            assert torch.allclose(roi_align(fm, rois, (5, 4), 0.2, sr, al),
                                  tv_roi_align(fm, rois, (5, 4), 0.2, sr, al), atol=1e-8), (sr, al)
    q = rois.clone(); q[:, 1:] = (q[:, 1:] * 4).round() / 4      # quarter-pixel boxes
    assert torch.equal(roi_pool(fm, q, (3, 3), 0.25), tv_roi_pool(fm, q, (3, 3), 0.25))
    print("matches torchvision.ops.roi_align / roi_pool")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("roi-align", roi_align=roi_align, roi_pool=roi_pool)